In [23]:
%%writefile solvers.py
import numpy as np

class SORSolver:
    """
    SOR and PSOR solvers for tridiagonal systems A x = d.

    SOR  : standard solver for European options
    PSOR : projected solver for American options (enforces V >= payoff)
    """

    def __init__(self, omega=1.2, tol=1e-8, max_iter=10_000):
        """
        omega    : relaxation factor (1 = Gauss-Seidel, 1 < omega < 2 = over-relaxation)
        tol      : convergence tolerance
        max_iter : maximum number of iterations
        """
        self.omega    = omega
        self.tol      = tol
        self.max_iter = max_iter

    def solve(self, a, b, c, d, x0=None) -> np.ndarray:
        """Solve A x = d and return solution vector x."""

        n = len(b)

        # Start from x0 if provided, otherwise start from zeros
        x = np.zeros(n) if x0 is None else x0.copy()

        for _ in range(self.max_iter):

            x_old = x.copy()  # snapshot of x before this iteration

            for i in range(n):

                # Contributions from left and right neighbours
                left  = a[i-1] * x[i-1]     if i > 0     else 0.0
                right = c[i]   * x_old[i+1] if i < n - 1 else 0.0

                # Gauss-Seidel update, then blend with omega (SOR step)
                x_gs = (d[i] - left - right) / b[i]
                x[i] = (1 - self.omega) * x_old[i] + self.omega * x_gs

            # Converged if largest change across all elements is below tolerance
            if np.max(np.abs(x - x_old)) < self.tol:
                return x

        raise RuntimeError("SOR did not converge within max_iter.")

    def solve_projected(self, a, b, c, d, payoff, x0=None) -> np.ndarray:
        """
        Solve A x = d with early exercise constraint x >= payoff.
        Used for American options.
        """

        n = len(b)

        # Start from payoff if no initial guess provided
        x = payoff.copy() if x0 is None else x0.copy()

        for _ in range(self.max_iter):

            x_old = x.copy()  # snapshot of x before this iteration

            for i in range(n):

                # Contributions from left and right neighbours
                left  = a[i-1] * x[i-1]     if i > 0     else 0.0
                right = c[i]   * x_old[i+1] if i < n - 1 else 0.0

                # Gauss-Seidel update, then blend with omega (SOR step)
                x_gs    = (d[i] - left - right) / b[i]
                relaxed = (1 - self.omega) * x_old[i] + self.omega * x_gs

                # Projection step — enforce early exercise constraint
                x[i] = max(payoff[i], relaxed)

            # Converged if largest change across all elements is below tolerance
            if np.max(np.abs(x - x_old)) < self.tol:
                return x

        raise RuntimeError("PSOR did not converge within max_iter.")

Writing solvers.py


In [24]:
%%writefile pricers.py
import numpy as np
import time
from abc import ABC, abstractmethod
from scipy.stats import norm
from solvers import SORSolver

class OptionPricer(ABC):
    """
    Abstract base class for all option pricers.
    Holds shared inputs, payoff helper, plotting, and repr.
    All subclasses must implement solve().
    """

    def __init__(
        self,
        option_type:  str,
        S0:    float,
        K:     float,
        T:     float,
        r:     float,
        sigma: float,
        S_max: float = 200.0,
        M:     int   = 500,
    ):
        if option_type.lower() not in ("call", "put"):
            raise ValueError("option_type must be 'call' or 'put'")

        self.option_type = option_type.lower()
        self.S0    = S0
        self.K     = K
        self.T     = T
        self.r     = r
        self.sigma = sigma
        self.S_max = S_max
        self.M     = M

        self.result = None   # populated after .solve()

    def _payoff(self, S: np.ndarray) -> np.ndarray:
        """Intrinsic value across asset grid — shared by all pricers."""
        if self.option_type == "call":
            return np.maximum(S - self.K, 0.0)
        return np.maximum(self.K - S, 0.0)

    @abstractmethod
    def solve(self) -> "OptionPricer":
        """Price the option. Must be implemented by every subclass."""
        pass

    def price_at(self, S):
        if self.result is None:
            raise RuntimeError("Call .solve() first")
        return np.interp(
            S,
            self.result["S_grid"],
            self.result["V_grid"][:, -1]
        )

    def plot(self) -> None:
        """Plot option value and Greeks — shared by FD and BS pricers."""
        import matplotlib.pyplot as plt

        if self.result is None:
            raise RuntimeError("Call .solve() before .plot()")

        r   = self.result
        pad = 5
        S_int = r["S_grid"][pad:-pad]

        fig, axs = plt.subplots(2, 2, figsize=(10, 8))
        fig.suptitle(
            f"{self.__class__.__name__} — {self.option_type.capitalize()} at t=0",
            fontsize=14
        )

        plots = [
            (axs[0, 0], r["S_grid"],  r["V_grid"][:, -1],          "Option Value", "Value"),
            (axs[0, 1], S_int, r["delta_grid"][pad:-pad], "Delta",  "Delta"),
            (axs[1, 0], S_int, r["gamma_grid"][pad:-pad], "Gamma",  "Gamma"),
            (axs[1, 1], S_int, r["theta_grid"][pad:-pad], "Theta",  "Theta (per year)"),
        ]
        for ax, x, y, title, ylabel in plots:
            ax.plot(x, y)
            ax.set_title(title)
            ax.set_xlabel("Asset Price S")
            ax.set_ylabel(ylabel)
            ax.grid(True)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
        
    def greeks_at(self, S):
        if self.result is None:
            raise RuntimeError("Call .solve() first")

        S_grid = self.result["S_grid"]

        return {
            "delta": np.interp(S, S_grid, self.result["delta_grid"]),
            "gamma": np.interp(S, S_grid, self.result["gamma_grid"]),
            "theta": np.interp(S, S_grid, self.result["theta_grid"]),
        }

    def __repr__(self) -> str:
        status = f"price={self.result['price']:.4f}" if self.result else "unsolved"
        return (
            f"{self.__class__.__name__}("
            f"{self.option_type}, S0={self.S0}, K={self.K}, "
            f"T={self.T}, r={self.r}, sigma={self.sigma} | {status})"
        )

Writing pricers.py


In [25]:
%%writefile black_scholes.py
from pricers import OptionPricer
import numpy as np
import time
from scipy.stats import norm

class BlackScholesPricer(OptionPricer):
    """
    Analytical closed-form Black-Scholes pricer.
    European options only.
    No __init__ needed — base class handles all inputs.
    """

    def solve(self) -> "BlackScholesPricer":
        start = time.time()

        S  = np.linspace(0.0001, self.S_max, self.M + 1)  # avoid log(0)
        dS = S[1] - S[0]
        V  = np.zeros((self.M + 1, 2))
        delta_grid = np.zeros(self.M + 1)
        gamma_grid = np.zeros(self.M + 1)
        theta_grid = np.zeros(self.M + 1)

        if self.T <= 0:
            # At expiry — value equals payoff
            V[:, -1] = self._payoff(S)
        else:
            d1 = (np.log(S / self.K) + (self.r + 0.5 * self.sigma**2) * self.T) \
                 / (self.sigma * np.sqrt(self.T))
            d2     = d1 - self.sigma * np.sqrt(self.T)
            pdf_d1 = norm.pdf(d1)

            if self.option_type == "call":
                V[:, -1]   = S * norm.cdf(d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
                delta_grid = norm.cdf(d1)
                theta_grid = (
                    -S * pdf_d1 * self.sigma / (2 * np.sqrt(self.T))
                    - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
                )
            else:
                V[:, -1]   = self.K * np.exp(-self.r * self.T) * norm.cdf(-d2) - S * norm.cdf(-d1)
                delta_grid = norm.cdf(d1) - 1
                theta_grid = (
                    -S * pdf_d1 * self.sigma / (2 * np.sqrt(self.T))
                    + self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(-d2)
                )

            gamma_grid = pdf_d1 / (S * self.sigma * np.sqrt(self.T))

        self.result = {
            "price":        float(np.interp(self.S0, S, V[:, -1])),
            "delta":        float(np.interp(self.S0, S, delta_grid)),
            "gamma":        float(np.interp(self.S0, S, gamma_grid)),
            "theta":        float(np.interp(self.S0, S, theta_grid)),
            "S_grid":       S,
            "V_grid":       V,
            "delta_grid":   delta_grid,
            "gamma_grid":   gamma_grid,
            "theta_grid":   theta_grid,
            "dt":           None,
            "dS":           dS,
            "T":            self.T,
            "M":            self.M,
            "N":            None,
            "scheme":       "closed_form",
            "option_type":  self.option_type,
            "option_style": "european",
            "time_taken":   time.time() - start,
        }
        return self

Writing black_scholes.py


In [26]:
%%writefile finite_difference.py
from pricers import OptionPricer
import numpy as np
import time
from scipy.stats import norm
from solvers import SORSolver

class FiniteDifferencePricer(OptionPricer):
    """
    Finite difference pricer for the Black-Scholes PDE.
    Supports explicit and implicit schemes, European and American styles.
    """

    def __init__(
        self,
        option_type:  str,
        option_style: str,
        scheme:       str,
        S0:    float,
        K:     float,
        T:     float,
        r:     float,
        sigma: float,
        S_max: float,
        M:     int,
        N:     int,
    ):
        super().__init__(option_type, S0, K, T, r, sigma, S_max, M)

        if option_style.lower() not in ("european", "american"):
            raise ValueError("option_style must be 'european' or 'american'")
        if scheme.lower() not in ("explicit", "implicit"):
            raise ValueError("scheme must be 'explicit' or 'implicit'")

        self.option_style = option_style.lower()
        self.scheme       = scheme.lower()
        self.N            = N
        self._solver      = SORSolver()  # reused across all time steps

    def _apply_boundary(self, V, k, dt) -> None:
        """Set boundary conditions at S=0 and S=S_max."""
        tau = (k + 1) * dt
        if self.option_type == "call":
            V[0, k + 1]      = 0.0
            V[self.M, k + 1] = self.S_max - self.K * np.exp(-self.r * tau)
        else:
            V[0, k + 1]      = self.K * np.exp(-self.r * tau)
            V[self.M, k + 1] = 0.0

    def _solve_explicit(self, V, S, dt, dS) -> None:
        """Explicit finite difference time stepping."""
        stability_limit = dS**2 / (self.sigma**2 * self.S_max**2)
        if dt > stability_limit:
            raise ValueError("Explicit scheme unstable: reduce dt or increase M.")

        for k in range(self.N):
            for i in range(1, self.M):
                a = 0.5 * dt * (self.sigma**2 * i**2 - self.r * i)
                b = 1.0 - dt * (self.sigma**2 * i**2 + self.r)
                c = 0.5 * dt * (self.sigma**2 * i**2 + self.r * i)

                V_cont = a * V[i - 1, k] + b * V[i, k] + c * V[i + 1, k]

                if self.option_style == "american":
                    intrinsic  = S[i] - self.K if self.option_type == "call" else self.K - S[i]
                    V[i, k+1] = max(V_cont, intrinsic)
                else:
                    V[i, k+1] = V_cont

            self._apply_boundary(V, k, dt)

    def _solve_implicit(self, V, S, dt) -> None:
        """Implicit (backward Euler) finite difference time stepping."""
        for k in range(self.N):
            self._apply_boundary(V, k, dt)

            i      = np.arange(1, self.M)
            a_full = -0.5 * dt * (self.sigma**2 * i**2 - self.r * i)
            b      =  1.0 + dt * (self.sigma**2 * i**2 + self.r)
            c_full = -0.5 * dt * (self.sigma**2 * i**2 + self.r * i)

            rhs = V[1:self.M, k].copy()
            rhs[0]  -= a_full[0]  * V[0, k + 1]
            rhs[-1] -= c_full[-1] * V[self.M, k + 1]

            if self.option_style == "european":
                V[1:self.M, k+1] = self._solver.solve(
                    a_full[1:], b, c_full[:-1], rhs, x0=V[1:self.M, k]
                )
            else:
                payoff = self._payoff(S[1:self.M])
                V[1:self.M, k+1] = self._solver.solve_projected(
                    a_full[1:], b, c_full[:-1], rhs, payoff, x0=V[1:self.M, k]
                )

    def _compute_greeks(self, V, S, dS, dt) -> tuple:
        """Compute point and grid Greeks at t=0."""
        i0    = np.clip(np.searchsorted(S, self.S0), 1, self.M - 1)
        price = np.interp(self.S0, S, V[:, -1])
        delta = (V[i0 + 1, -1] - V[i0 - 1, -1]) / (2.0 * dS)
        gamma = (V[i0 + 1, -1] - 2.0 * V[i0, -1] + V[i0 - 1, -1]) / dS**2
        theta = -(V[i0, -1] - V[i0, -2]) / dt

        delta_grid = np.zeros(self.M + 1)
        gamma_grid = np.zeros(self.M + 1)
        theta_grid = np.zeros(self.M + 1)

        for i in range(1, self.M):
            delta_grid[i] = (V[i + 1, -1] - V[i - 1, -1]) / (2.0 * dS)
            gamma_grid[i] = (V[i + 1, -1] - 2.0 * V[i, -1] + V[i - 1, -1]) / dS**2
            theta_grid[i] = -(V[i, -1] - V[i, -2]) / dt

        return price, delta, gamma, theta, delta_grid, gamma_grid, theta_grid

    def solve(self) -> "FiniteDifferencePricer":
        start = time.time()
        dS = self.S_max / self.M
        dt = self.T / self.N
        S  = np.linspace(0, self.S_max, self.M + 1)
        V  = np.zeros((self.M + 1, self.N + 1))

        V[:, 0] = self._payoff(S)   # terminal condition

        if self.scheme == "explicit":
            self._solve_explicit(V, S, dt, dS)
        else:
            self._solve_implicit(V, S, dt)

        price, delta, gamma, theta, dg, gg, tg = self._compute_greeks(V, S, dS, dt)

        self.result = {
            "price":        price,
            "delta":        delta,
            "gamma":        gamma,
            "theta":        theta,
            "S_grid":       S,
            "V_grid":       V,
            "delta_grid":   dg,
            "gamma_grid":   gg,
            "theta_grid":   tg,
            "dt":           dt,
            "dS":           dS,
            "T":            self.T,
            "M":            self.M,
            "N":            self.N,
            "scheme":       self.scheme,
            "option_type":  self.option_type,
            "option_style": self.option_style,
            "time_taken":   time.time() - start,
        }
        return self

Writing finite_difference.py


In [27]:
%%writefile crank_nicolson.py
from pricers import OptionPricer
import numpy as np
import time
from scipy.stats import norm
from solvers import SORSolver

class CrankNicolsonPricer(OptionPricer):
    """
    Crank-Nicolson finite difference pricer.
    Second-order accurate in both time and space.
    Supports European and American options.
    """

    def __init__(
        self,
        option_type:  str,
        option_style: str,
        S0:    float,
        K:     float,
        T:     float,
        r:     float,
        sigma: float,
        S_max: float,
        M:     int,
        N:     int,
        omega: float = 1.2,
    ):
        super().__init__(option_type, S0, K, T, r, sigma, S_max, M)

        if option_style.lower() not in ("european", "american"):
            raise ValueError("option_style must be 'european' or 'american'")

        self.option_style = option_style.lower()
        self.N            = N
        self._solver      = SORSolver(omega=omega)

    def _apply_boundary(self, V_new, n, dt) -> None:
        """Set boundary conditions at S=0 and S=S_max."""
        tau = self.T - n * dt
        if self.option_type == "call":
            V_new[0]  = 0.0
            V_new[-1] = self.S_max - self.K * np.exp(-self.r * tau)
        else:
            V_new[0]  = self.K * np.exp(-self.r * tau)
            V_new[-1] = 0.0

    def _build_lhs_rhs(self, V_old, S, dS, dt) -> tuple:
        """Build tridiagonal LHS and RHS for the CN system."""
        i  = np.arange(1, self.M)
        Si = S[i]

        # CN coefficients — average of explicit and implicit contributions
        alpha = 0.25 * dt * (self.sigma**2 * Si**2 / dS**2 - self.r * Si / dS)
        beta  = -0.5 * dt * (self.sigma**2 * Si**2 / dS**2 + self.r)
        gamma = 0.25 * dt * (self.sigma**2 * Si**2 / dS**2 + self.r * Si / dS)

        a = -alpha[1:]    # sub-diagonal
        b =  1 - beta     # main diagonal
        c = -gamma[:-1]   # super-diagonal

        rhs = (
            alpha * V_old[:-2]
            + (1 + beta) * V_old[1:-1]
            + gamma * V_old[2:]
        )
        return a, b, c, rhs

    def _compute_greeks_grid(self, V, S, dS) -> tuple:
        """Compute Greeks across full spatial grid."""
        delta_grid = np.zeros(self.M + 1)
        gamma_grid = np.zeros(self.M + 1)
        theta_grid = np.zeros(self.M + 1)

        for i in range(1, self.M):
            delta_grid[i] = (V[i + 1] - V[i - 1]) / (2.0 * dS)
            gamma_grid[i] = (V[i + 1] - 2.0 * V[i] + V[i - 1]) / dS**2
            # Theta from BS PDE (CN-consistent)
            theta_grid[i] = (
                -0.5 * self.sigma**2 * S[i]**2 * gamma_grid[i]
                - self.r * S[i] * delta_grid[i]
                + self.r * V[i]
            )
        return delta_grid, gamma_grid, theta_grid

    def solve(self) -> "CrankNicolsonPricer":
        start = time.time()

        dS    = self.S_max / self.M
        dt    = self.T / self.N
        S     = np.linspace(0, self.S_max, self.M + 1)
        V_old = self._payoff(S)
        V_new = np.zeros(self.M + 1)

        # Backward time-stepping
        for n in range(self.N):
            self._apply_boundary(V_new, n, dt)

            a, b, c, rhs = self._build_lhs_rhs(V_old, S, dS, dt)
            rhs[0]  -= a[0]  * V_new[0]
            rhs[-1] -= c[-1] * V_new[-1]

            if self.option_style == "european":
                V_new[1:-1] = self._solver.solve(a, b, c, rhs, x0=V_old[1:-1])
            else:
                payoff      = self._payoff(S[1:-1])
                V_new[1:-1] = self._solver.solve_projected(a, b, c, rhs, payoff, x0=V_old[1:-1])

            V_old[:] = V_new[:]  # roll forward

        # Point Greeks at S0
        idx       = np.clip(np.searchsorted(S, self.S0), 1, self.M - 1)
        price     = np.interp(self.S0, S, V_old)
        delta     = (V_old[idx + 1] - V_old[idx - 1]) / (2.0 * dS)
        gamma_val = (V_old[idx + 1] - 2.0 * V_old[idx] + V_old[idx - 1]) / dS**2
        theta     = (
            -0.5 * self.sigma**2 * S[idx]**2 * gamma_val
            - self.r * S[idx] * delta
            + self.r * V_old[idx]
        )

        delta_grid, gamma_grid, theta_grid = self._compute_greeks_grid(V_old, S, dS)

        # Wrap in 2D so base class plot() works consistently
        V_grid        = np.zeros((self.M + 1, 2))
        V_grid[:, -1] = V_old

        self.result = {
            "price":        price,
            "delta":        delta,
            "gamma":        gamma_val,
            "theta":        theta,
            "S_grid":       S,
            "V_grid":       V_grid,
            "delta_grid":   delta_grid,
            "gamma_grid":   gamma_grid,
            "theta_grid":   theta_grid,
            "dt":           dt,
            "dS":           dS,
            "T":            self.T,
            "M":            self.M,
            "N":            self.N,
            "scheme":       "crank_nicolson",
            "option_type":  self.option_type,
            "option_style": self.option_style,
            "time_taken":   time.time() - start,
        }
        return self

Writing crank_nicolson.py


In [32]:
%%writefile monte_carlo.py
from pricers import OptionPricer
import numpy as np
import time


class MonteCarloPricer(OptionPricer):
    """
    Monte Carlo pricer for European options.

    Features:
        - Common Random Numbers (CRN) for stable Greeks
        - Pathwise Delta estimator
        - Central bump Gamma and Theta
        - Pseudo spatial grid for plotting
        - 6-panel plot output
    """

    def __init__(
        self,
        option_type: str,
        S0:       float,
        K:        float,
        T:        float,
        r:        float,
        sigma:    float,
        n_paths:  int,
        n_steps:  int   = 100,
        eps_S:    float = 1e-2,   # spot bump size for Gamma
        eps_T:    float = 1e-4,   # time bump size for Theta
    ):
        # Pass shared parameters up to OptionPricer base class
        super().__init__(option_type, S0, K, T, r, sigma)
        self.n_paths = n_paths
        self.n_steps = n_steps
        self.eps_S   = eps_S
        self.eps_T   = eps_T

    # ================================================================
    # Private helpers
    # ================================================================

    def _simulate_terminal(self, S0: float, T: float, Z: np.ndarray) -> np.ndarray:
        """
        Simulate terminal asset prices S(T) using the GBM closed form.

        S(T) = S0 * exp( (r - 0.5σ²)T + σ√T Z )

        Accepts external Z so the same random numbers can be reused
        across bumped scenarios (Common Random Numbers).
        """
        return S0 * np.exp(
            (self.r - 0.5 * self.sigma**2) * T
            + self.sigma * np.sqrt(T) * Z
        )

    def _simulate_paths(self, Z_paths: np.ndarray) -> np.ndarray:
        """
        Simulate full asset price paths using Euler-Maruyama discretisation.
        Used only for the paths plot — not for pricing or Greeks.

        Returns S of shape (n_paths, n_steps + 1).
        """
        dt = self.T / self.n_steps
        S  = np.zeros((self.n_paths, self.n_steps + 1))
        S[:, 0] = self.S0

        for t in range(1, self.n_steps + 1):
            S[:, t] = S[:, t - 1] * np.exp(
                (self.r - 0.5 * self.sigma**2) * dt
                + self.sigma * np.sqrt(dt) * Z_paths[:, t - 1]
            )
        return S

    def _point_greeks(self, s: float, discount: float, Z: np.ndarray) -> tuple:
        """
        Compute price, Delta, Gamma, Theta at a single spot value s.
        Reuses Z (CRN) for all bump calculations so noise cancels cleanly.
        """
        # Simulate terminal prices at s, s+eps, s-eps using same Z
        ST_s      = self._simulate_terminal(s,              self.T, Z)
        ST_s_up   = self._simulate_terminal(s + self.eps_S, self.T, Z)
        ST_s_down = self._simulate_terminal(s - self.eps_S, self.T, Z)

        # Discounted expected payoffs
        price_s    = discount * np.mean(self._payoff(ST_s))
        price_s_up = discount * np.mean(self._payoff(ST_s_up))
        price_s_dn = discount * np.mean(self._payoff(ST_s_down))

        # Pathwise Delta — differentiates payoff through indicator function
        indicator = (ST_s > self.K).astype(float) if self.option_type == "call" \
                    else (ST_s < self.K).astype(float)
        delta_s = discount * np.mean(indicator * ST_s / s)
        if self.option_type == "put":
            delta_s *= -1

        # Gamma — central finite difference on price w.r.t. spot
        gamma_s = (price_s_up - 2 * price_s + price_s_dn) / self.eps_S**2

        # Theta — derived from BS PDE so it's consistent with Delta and Gamma
        # ∂V/∂t = -0.5σ²S²Γ - rSΔ + rV
        theta_s = (
            -0.5 * self.sigma**2 * s**2 * gamma_s
            - self.r * s * delta_s
            + self.r * price_s
        )

        return price_s, delta_s, gamma_s, theta_s

    # ================================================================
    # solve()
    # ================================================================

    def solve(self) -> "MonteCarloPricer":
        start = time.time()

        discount = np.exp(-self.r * self.T)

        # ----------------------------------------------------------------
        # ONE set of random numbers shared across ALL calculations (CRN)
        # This ensures bumped scenarios differ only in S0/T — not in noise
        # ----------------------------------------------------------------
        Z = np.random.randn(self.n_paths)

        # ----------------------------------------------------------------
        # Point estimates at S0
        # ----------------------------------------------------------------

        # Terminal prices at base spot — renamed ST_base to protect
        # from being overwritten inside the pseudo-grid loop below
        ST_base = self._simulate_terminal(self.S0, self.T, Z)
        price   = discount * np.mean(self._payoff(ST_base))

        # Delta — pathwise estimator
        indicator = (ST_base > self.K).astype(float) if self.option_type == "call" \
                    else (ST_base < self.K).astype(float)
        delta = discount * np.mean(indicator * ST_base / self.S0)
        if self.option_type == "put":
            delta *= -1

        # Gamma — central bump using CRN
        ST_up      = self._simulate_terminal(self.S0 + self.eps_S, self.T, Z)
        ST_down    = self._simulate_terminal(self.S0 - self.eps_S, self.T, Z)
        price_up   = discount * np.mean(self._payoff(ST_up))
        price_down = discount * np.mean(self._payoff(ST_down))
        gamma      = (price_up - 2 * price + price_down) / self.eps_S**2

        # Theta — central bump in time using CRN
        ST_T_plus     = self._simulate_terminal(self.S0, self.T + self.eps_T, Z)
        ST_T_minus    = self._simulate_terminal(self.S0, self.T - self.eps_T, Z)
        price_T_plus  = np.exp(-self.r * (self.T + self.eps_T)) * np.mean(self._payoff(ST_T_plus))
        price_T_minus = np.exp(-self.r * (self.T - self.eps_T)) * np.mean(self._payoff(ST_T_minus))
        theta         = (price_T_minus - price_T_plus) / (2 * self.eps_T)

        # ----------------------------------------------------------------
        # Pseudo spatial grid — price and Greeks across range of spot values
        # Used for plotting only, not for the point estimates above
        # Same Z reused across all grid points (CRN)
        # ----------------------------------------------------------------
        S_grid     = np.linspace(1, self.S_max, 100)
        V_grid     = []
        delta_grid = []
        gamma_grid = []
        theta_grid = []

        for s in S_grid:
            price_s, delta_s, gamma_s, theta_s = self._point_greeks(s, discount, Z)
            V_grid.append(price_s)
            delta_grid.append(delta_s)
            gamma_grid.append(gamma_s)
            theta_grid.append(theta_s)

        # ----------------------------------------------------------------
        # Simulate full paths — uses fresh randomness (for realistic plot)
        # Separate from Z so the paths aren't constrained to CRN structure
        # ----------------------------------------------------------------
        Z_paths = np.random.randn(self.n_paths, self.n_steps)
        S_paths = self._simulate_paths(Z_paths)

        self.result = {
            # Point estimates at S0
            "price":      float(price),
            "delta":      float(delta),
            "gamma":      float(gamma),
            "theta":      float(theta),

            # Spatial grid (for plotting)
            "S_grid":     S_grid,
            "V_grid":     np.array(V_grid).reshape(-1, 1),
            "delta_grid": np.array(delta_grid),
            "gamma_grid": np.array(gamma_grid),
            "theta_grid": np.array(theta_grid),

            # Path data (for plotting)
            "S_paths":    S_paths,
            "S_terminal": ST_base,

            "time_taken": time.time() - start,
        }
        return self

    # ================================================================
    # 6-panel plot
    # ================================================================

    def plot(self) -> None:
        """
        6-panel plot:
            [0,0] Option Value      [0,1] Delta
            [1,0] Gamma             [1,1] Theta
            [2,0] Simulated Paths   [2,1] Terminal Distribution
        """
        import matplotlib.pyplot as plt

        if self.result is None:
            raise RuntimeError("Call .solve() before .plot()")

        r         = self.result
        pad       = min(5, len(r["S_grid"]) // 10)
        S_int     = r["S_grid"][pad:-pad]
        S_paths   = r["S_paths"]
        ST        = r["S_terminal"]
        time_grid = np.linspace(0, self.T, self.n_steps + 1)

        fig, axs = plt.subplots(3, 2, figsize=(12, 14))
        fig.suptitle(
            f"MonteCarloPricer — {self.option_type.capitalize()} Option",
            fontsize=14
        )

        # Panel definitions — (ax, x data, y data, title, ylabel)
        panels = [
            (axs[0, 0], r["S_grid"], r["V_grid"][:, -1],          "Option Value", "Value"),
            (axs[0, 1], S_int,       r["delta_grid"][pad:-pad],    "Delta",        "Delta"),
            (axs[1, 0], S_int,       r["gamma_grid"][pad:-pad],    "Gamma",        "Gamma"),
            (axs[1, 1], S_int,       r["theta_grid"][pad:-pad],    "Theta",        "Theta (per year)"),
        ]
        for ax, x, y, title, ylabel in panels:
            ax.plot(x, y)
            ax.set_title(title)
            ax.set_xlabel("Asset Price S")
            ax.set_ylabel(ylabel)
            ax.grid(True)

        # Simulated paths — plot first 50 to avoid clutter
        for i in range(min(50, self.n_paths)):
            axs[2, 0].plot(time_grid, S_paths[i], alpha=0.1, linewidth=0.8)
        axs[2, 0].axhline(self.K, linestyle="--", color="black", label=f"Strike K={self.K}")
        axs[2, 0].set_title("Simulated Paths")
        axs[2, 0].set_xlabel("Time")
        axs[2, 0].set_ylabel("Asset Price S")
        axs[2, 0].legend()
        axs[2, 0].grid(True)

        # Terminal price distribution
        axs[2, 1].hist(ST, bins=50, density=True, color="steelblue", alpha=0.7)
        axs[2, 1].axvline(self.K, linestyle="--", color="black", label=f"Strike K={self.K}")
        axs[2, 1].set_title("Terminal Distribution")
        axs[2, 1].set_xlabel("S(T)")
        axs[2, 1].set_ylabel("Density")
        axs[2, 1].legend()
        axs[2, 1].grid(True)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

Overwriting monte_carlo.py
